# Color 01 - Palette foundation (Solution)

Muc tieu: map dung loai du lieu vao loai color scale.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "owid_co2_subset.csv").exists():
            return p
    raise FileNotFoundError("Cannot locate data/owid_co2_subset.csv")

root = resolve_repo_root()
df = pd.read_csv(root / "data" / "owid_co2_subset.csv")
df = df[df["iso_code"].astype(str).str.len() == 3].copy()
d_recent = df[df["year"] >= 2000].copy()
latest_year = (
    d_recent.dropna(subset=["co2_per_capita", "gdp", "population"])
    ["year"]
    .max()
)
d_latest = d_recent[d_recent["year"] == latest_year].copy()
d_latest.head()

## 1) Categorical palette cho bien nominal (`continent`)

Chart phu hop: so sanh giua cac nhom roi rac (khong co thu tu tu nhien).
Mau co vai tro **phan biet nhom** (khong truyen tai magnitude).

In [ ]:
cat_df = (
    d_latest.groupby("country", as_index=False)
    .agg(co2_pc=("co2_per_capita", "mean"), pop=("population", "mean"))
    .nlargest(12, "pop")
    .sort_values("co2_pc", ascending=False)
)

plt.figure(figsize=(9, 4.5))
sns.barplot(
    data=cat_df,
    x="country",
    y="co2_pc",
    hue="country",
    palette="Set3",
    dodge=False,
    legend=False,
)
plt.title("Categorical palette: distinguish nominal groups (top population countries)")
plt.ylabel("CO2 per capita")
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 2) Sequential palette cho bien lien tuc (`gdpPercap`)

Chart phu hop: heatmap/choropleth hoac bang co thu tu.
Mau co vai tro **truyen tai muc do** (light -> dark).

In [ ]:
seq_df = (
    d_latest.groupby("country", as_index=False)
    .agg(co2_pc=("co2_per_capita", "mean"))
    .nlargest(20, "co2_pc")
    .sort_values("co2_pc", ascending=False)
)

norm = plt.Normalize(seq_df["co2_pc"].min(), seq_df["co2_pc"].max())
cmap = plt.cm.Blues
colors = cmap(norm(seq_df["co2_pc"].to_numpy()))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(seq_df["country"], seq_df["co2_pc"], color=colors)
ax.set_title("Sequential palette: magnitude encoding (CO2 per capita)")
ax.set_xlabel("CO2 per capita")
ax.invert_yaxis()
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="CO2 per capita")
plt.tight_layout(); plt.show()

## 3) Diverging palette khi co diem giua co y nghia (0)

Ta tao bien `lifeExp_delta` = lifeExp(country) - median(2007).
Chart phu hop: bar theo do lech dau (+/-) quanh moc trung tam.

In [ ]:
global_mean = d_latest["co2_per_capita"].mean()
d_latest["co2_pc_delta"] = d_latest["co2_per_capita"] - global_mean
focus = pd.concat(
    [d_latest.nsmallest(10, "co2_pc_delta"), d_latest.nlargest(10, "co2_pc_delta")],
    ignore_index=True,
).sort_values("co2_pc_delta")

v = focus["co2_pc_delta"].to_numpy()
norm = plt.Normalize(v.min(), v.max())
cmap = plt.cm.RdBu_r
colors = cmap(norm(v))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(focus["country"], focus["co2_pc_delta"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Diverging palette around meaningful center (global mean)")
ax.set_xlabel("co2_per_capita - global_mean")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Delta CO2 per capita")
plt.tight_layout(); plt.show()

## Reflection
- Vi sao khong dung palette categorical cho bien lien tuc?
- Neu khong co moc 0/target ro rang, ban co dung diverging khong?